In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict

from mad.objs.constants import G0
from mad.objs.common_schemas import MovableObject

In [17]:
class Payload(MovableObject):
    mass: float # kg
    area: float # m^2
    yield_kt: float # kt


@dataclass
class StageConfig:
    dry_mass: float  # kg
    propellant_mass: float  # kg
    thrust: float # N / s
    Isp: float  # s
    area: float # m^2
    Cd: float
    time_ECO: float # s
    time_sep: float  # s
    payload: Payload | None = None
    name: str = "Stage"

    @property
    def to_dict(self):
        return asdict(self)

In [ ]:
class MissileStage:

    def __init__(self, cfg: StageConfig):
        self.config = cfg
        self.dry_mass = cfg.dry_mass
        self.propellant_mass = cfg.propellant_mass

        self.thrust = cfg.thrust
        self.Isp = cfg.Isp

        self.area = cfg.area
        self.Cd = cfg.Cd

        self.exhaust_velocity = cfg.Isp * G0
        self.mass_flow_rate = cfg.thrust / self.exhaust_velocity

        self.active: bool = True
        self.payload = cfg.payload

    @property
    def mass(self)->float:
        payload_mass = self.payload.mass if self.payload else 0.0
        return self.dry_mass + self.propellant_mass + payload_mass

    def update(self, dt: float)->None:
        if not self.active:
            return

        if self.propellant_mass > 0:
            dm = self.mass_flow_rate * dt
            self.propellant_mass = max(0.0, self.propellant_mass - dm)
        else:
            self.active = False

    def thrust_force(self, direction: NDArray) -> NDArray:
        if self.propellant_mass > 0:
            return self.thrust * direction
        return np.zeros_like(direction)

In [ ]:
class Missile(MovableObject):
    def __init__(self, stages: list[MissileStage]):
        self.stages = stages

    @property
    def mass(self):
        return sum(stage.mass for stage in self.stages)

    @property
    def area(self):
        sum(stage.area for stage in self.stages)